In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.transforms import ScaledTranslation

from scipy.stats import linregress

sys.path.append("../src")
from plot import create_scatter_correlation_plot

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']

plt.rcParams.update(
    {
        "font.family": "Times New Roman"
    }
)

output_dir = Path("../../results/HD_DIT_HAP_generationRAW/figures/thesis")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. 代时计算

In [ ]:
time_vs_generations = pd.DataFrame(
    {
        "Samples": ["NSKY1328-4"] * 6 + ["NSKY1328-7"] * 6 + ["NSKY1328-8"] * 6,
        "Time (h)": [0,8,14,22,30,38,0,5,15.5,23,31,39,0,5,14.7,22.3,30.7,38.7],
        "Generations":[0,0,2.124,5.387,8.604,11.983,0,0,2.487,5.688,8.981,12.427,0,0,2.327,6.027,9.406,13.161]
    }
)

fig, ax = plt.subplots(figsize=(AX_WIDTH+2, AX_HEIGHT))

for sample, group in time_vs_generations.groupby("Samples"):
    ax.scatter(group["Time (h)"], group["Generations"], marker='o', label=sample, alpha=0.5)

# regression
x_values = time_vs_generations.query("Generations > 0")["Time (h)"]
y_values = time_vs_generations.query("Generations > 0")["Generations"]
slope, intercept, r_value, p_value, std_err = linregress(x_values, y_values)
x_fit = np.linspace(9, x_values.max(), 100)
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, color='darkred', ls="--")
ax.text(0.05, 0.95, f"PCC = {r_value:.3f}\nR² = {r_value**2:.3f}", transform=ax.transAxes, fontsize=18, verticalalignment='top')
ax.text(0.5, 0.3, f"y = {slope:.3f} * x {intercept:.3f}", transform=ax.transAxes, fontsize=18, verticalalignment='top', color='darkred')

ax.set_xlabel("Time (h)")
ax.set_ylabel("Generations")
ax.set_title("Time vs Generations for Different Samples")
ax.legend()
plt.tight_layout()
plt.savefig(output_dir/"time_vs_generations.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# 2. PCR quality control

In [ ]:
LD_1328_7_PBL_PBR = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])

LD1328_7_NSK = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_7_YYS = pd.read_csv("../../results/Spore2YES6_1328/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_7_merged = pd.merge(LD1328_7_NSK, LD1328_7_YYS, left_index=True, right_index=True, suffixes=("_NSK", "_YYS"))

spike_in_data = pd.read_csv("../../results/Spikein/14_spikein_correlation/spike_in_results.tsv", sep="\t")
spike_in_data = spike_in_data.query("Sample != 'Spikein0'")

In [ ]:
fig, axes = plt.subplot_mosaic([["(a)", "(b)", "(c)"]], figsize=(AX_WIDTH*3, AX_HEIGHT+1))

create_scatter_correlation_plot(
    LD_1328_7_PBL_PBR["PBL"].values,
    LD_1328_7_PBL_PBR["PBR"].values,
    ax=axes["(a)"],
    xscale="log",
    yscale="log"
)

create_scatter_correlation_plot(
    LD1328_7_merged["Reads_NSK"],
    LD1328_7_merged["Reads_YYS"],
    ax=axes["(b)"],
    xscale="log",
    yscale="log"
)

ax = axes["(c)"]
for idx, (strain, strain_df) in enumerate(spike_in_data.groupby("Name")):

    X = strain_df["Relative_Dilution_Ratio"]
    Y = strain_df["Relative_Read_Ratio"]
    
    ax.scatter(X, Y, label=f"{strain}", facecolor="none", edgecolor=COLORS[idx], 
               s=150, lw=1.5, alpha=0.9)


slope, intercept, r_value, p_value, std_err = linregress(
    spike_in_data["Relative_Dilution_Ratio"],
    spike_in_data["Relative_Read_Ratio"]
)
r2_scores = r_value ** 2

line_x = np.array([-8, 0])
line_y = slope * line_x + intercept
ax.plot(line_x, line_y, color="black", ls="--", alpha=0.7, lw=2.5)

ax.set_xlabel("log$_{2}$(relative dilution ratio)")
ax.set_ylabel("log$_{2}$(relative read ratio)")
ax.set_xticks([-8, -6, -4, -2, 0])
ax.set_yticks([-8, -6, -4, -2, 0])
ax.set_xticklabels([-8, -6, -4, -2, 0])
ax.set_yticklabels([-8, -6, -4, -2, 0])

ax.text(0.05, 0.95, f"PCC={r_value:.2f}\nR²={r2_scores:.2f}\nSlope={slope:.2f}\nIntercept={intercept:.2f}", transform=ax.transAxes, ha="left", va="top")
ax.legend(loc="lower right", fontsize=14, frameon=False)

axes["(a)"].set_xlabel("PBL Reads")
axes["(a)"].set_ylabel("PBR Reads")

axes["(b)"].set_xlabel("Reads of Biological Replicate 1")
axes["(b)"].set_ylabel("Reads of Biological Replicate 2")

for label, ax in axes.items():
    ax.text(0, 1, label, transform=(ax.transAxes+ ScaledTranslation(-0.6, 0.4, fig.dpi_scale_trans)), fontsize=24, verticalalignment='top')

plt.tight_layout()
plt.savefig(output_dir/"PCR_quality_control.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()
